In [ ]:
# Ô 1 - cài môi trường (chép từ docs/kaggle-colab.md §4). Chạy lại an toàn.
%cd /content
![ -d vcf_repo ] || git clone -q https://github.com/Nguyendev037/VCuboidFit.git vcf_repo
!pip -q install spconv-cu120 "numpy<2" nuscenes-devkit easydict tensorboardX kornia==0.6.12     scikit-learn joblib pyarrow pandas pyyaml opencv-python-headless requests
![ -d OpenPCDet ] || (wget -qO pcdet.tgz https://github.com/open-mmlab/OpenPCDet/archive/233f849829b6ac19afb8af8837a0246890908755.tar.gz     && tar xzf pcdet.tgz && mv OpenPCDet-233f849829b6ac19afb8af8837a0246890908755 OpenPCDet     && echo 233f849829b6ac19afb8af8837a0246890908755 > OpenPCDet/BUILD_COMMIT     && python vcf_repo/model/docker/tier1/patch_pcdet.py OpenPCDet     && cd OpenPCDet && TORCH_CUDA_ARCH_LIST="7.5;8.0;8.6;8.9" python setup.py develop)

In [ ]:
# Ô 2 - lấy mã nguồn mới nhất
%cd /content/vcf_repo
!git pull -q
!pip -q install -e model/worker
import os
os.environ["PCDET_ROOT"] = "/content/OpenPCDet"
os.environ["PYTHONPATH"] = "/content/vcf_repo/model/worker:/content/OpenPCDet"

In [ ]:
# Ô 3 - nhập URL tunnel + token (token KHÔNG hiển thị, KHÔNG lưu vào notebook)
from getpass import getpass
SERVER = input("URL tunnel của worker (https://...): ").strip()
os.environ["VCF_REMOTE_TOKEN"] = getpass("VCF_REMOTE_TOKEN: ")

In [ ]:
# Ô 4 - chạy agent (vòng lặp kéo việc; dừng bằng nút Stop). Thêm --dry-run để thử không GPU.
import subprocess, sys
subprocess.run([sys.executable, "-u", "model/scripts/colab_agent.py", "--server", SERVER,
                "--work", "/content/vcf"], env=os.environ)